# Bakery Day Backtest

Backtest notebook for bakery-level daily forecast.

Inputs:
- `reports/bakery_day_model_holdout_predictions.csv`
- `reports/bakery_day_model_bias_by_bakery.csv`

The notebook compares:
- actual sales
- base holdout forecast
- bias-adjusted holdout forecast


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

HOLDOUT_PATH = ROOT / "reports" / "bakery_day_model_holdout_predictions.csv"
BIAS_PATH = ROOT / "reports" / "bakery_day_model_bias_by_bakery.csv"
BIAS_CLIP_PCT = 0.15

holdout = pd.read_csv(HOLDOUT_PATH, encoding="utf-8-sig")
bias = pd.read_csv(BIAS_PATH, encoding="utf-8-sig")

holdout["date"] = pd.to_datetime(holdout["date"], errors="coerce")
holdout["bakery_sales"] = pd.to_numeric(holdout["bakery_sales"], errors="coerce").fillna(0.0)
holdout["bakery_day_forecast"] = pd.to_numeric(holdout["bakery_day_forecast"], errors="coerce").fillna(0.0)

bias_small = bias[["bakery_id", "bias"]].copy()
bias_small["bias"] = pd.to_numeric(bias_small["bias"], errors="coerce").fillna(0.0)
bias_small = bias_small.groupby("bakery_id", as_index=False).agg(bias=("bias", "mean"))

backtest = holdout.merge(bias_small, on="bakery_id", how="left")
backtest["bias"] = backtest["bias"].fillna(0.0)
backtest["bias_adjustment_capped"] = backtest["bias"].clip(
    lower=-backtest["bakery_day_forecast"] * BIAS_CLIP_PCT,
    upper=backtest["bakery_day_forecast"] * BIAS_CLIP_PCT,
)
backtest["bakery_day_forecast_bias_adj"] = (
    backtest["bakery_day_forecast"] + backtest["bias_adjustment_capped"]
).clip(lower=0.0)

backtest["error_base"] = backtest["bakery_sales"] - backtest["bakery_day_forecast"]
backtest["error_bias_adj"] = backtest["bakery_sales"] - backtest["bakery_day_forecast_bias_adj"]
backtest = backtest.sort_values(["bakery_id", "date"]).reset_index(drop=True)

print(f"rows={len(backtest):,} | dates={backtest['date'].nunique()} | bakeries={backtest['bakery_id'].nunique()}")
display(backtest.head())


In [ ]:
def metrics_frame(df: pd.DataFrame, actual_col: str, pred_cols: list[str]) -> pd.DataFrame:
    rows = []
    actual = pd.to_numeric(df[actual_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    for pred_col in pred_cols:
        pred = pd.to_numeric(df[pred_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        rows.append(
            {
                "model": pred_col,
                "mae": float(np.mean(np.abs(actual - pred))),
                "mse": float(np.mean((actual - pred) ** 2)),
                "wmape": float(np.sum(np.abs(actual - pred)) / (np.sum(actual) + 1e-8) * 100),
                "bias": float(np.mean(actual - pred)),
                "forecast_total": float(pred.sum()),
                "actual_total": float(actual.sum()),
            }
        )
    return pd.DataFrame(rows)


overall_metrics = metrics_frame(
    backtest,
    actual_col="bakery_sales",
    pred_cols=["bakery_day_forecast", "bakery_day_forecast_bias_adj"],
)
display(overall_metrics)


In [ ]:
daily = (
    backtest.groupby("date", as_index=False)
    .agg(
        actual=("bakery_sales", "sum"),
        forecast_base=("bakery_day_forecast", "sum"),
        forecast_bias_adj=("bakery_day_forecast_bias_adj", "sum"),
    )
    .sort_values("date")
)
daily["error_base"] = daily["actual"] - daily["forecast_base"]
daily["error_bias_adj"] = daily["actual"] - daily["forecast_bias_adj"]
display(daily.head())


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily["date"], daily["actual"], marker="o", label="Actual")
ax.plot(daily["date"], daily["forecast_base"], marker="o", label="Forecast base")
ax.plot(daily["date"], daily["forecast_bias_adj"], marker="o", label="Forecast bias adj")
ax.set_title("Network-level bakery sales by day")
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.axhline(0.0, color="black", linewidth=1)
ax.plot(daily["date"], daily["error_base"], marker="o", label="Base error")
ax.plot(daily["date"], daily["error_bias_adj"], marker="o", label="Bias-adj error")
ax.set_title("Network-level daily error: actual - forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Error")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
bias_view = bias.copy().sort_values("abs_bias", ascending=False).reset_index(drop=True)
display(bias_view.head(20))


In [ ]:
def plot_bakery(bakery_id=None, bakery_name_contains=None):
    work = backtest.copy()
    if bakery_id is not None:
        work = work[work["bakery_id"] == bakery_id].copy()
    if bakery_name_contains is not None:
        work = work[work["bakery_name"].str.contains(bakery_name_contains, case=False, na=False)].copy()
    if work.empty:
        raise ValueError("No rows found for the requested bakery filter")

    bakery_name = work["bakery_name"].iloc[0]
    bakery_id_value = work["bakery_id"].iloc[0]
    city = work["city"].iloc[0]

    metrics = metrics_frame(
        work,
        actual_col="bakery_sales",
        pred_cols=["bakery_day_forecast", "bakery_day_forecast_bias_adj"],
    )
    display(metrics)

    fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
    axes[0].plot(work["date"], work["bakery_sales"], marker="o", label="Actual")
    axes[0].plot(work["date"], work["bakery_day_forecast"], marker="o", label="Forecast base")
    axes[0].plot(work["date"], work["bakery_day_forecast_bias_adj"], marker="o", label="Forecast bias adj")
    axes[0].set_title(f"Bakery {bakery_id_value} | {bakery_name} | {city}")
    axes[0].set_ylabel("Sales")
    axes[0].legend()

    axes[1].axhline(0.0, color="black", linewidth=1)
    axes[1].plot(work["date"], work["error_base"], marker="o", label="Base error")
    axes[1].plot(work["date"], work["error_bias_adj"], marker="o", label="Bias-adj error")
    axes[1].set_ylabel("Error")
    axes[1].set_xlabel("Date")
    axes[1].legend()

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# Examples: inspect the strongest shifted bakeries from holdout.
plot_bakery(bakery_id=262)
plot_bakery(bakery_id=245)


In [ ]:
bakery_metrics = []
for bakery_id, group in backtest.groupby("bakery_id", sort=False):
    metric_df = metrics_frame(group, "bakery_sales", ["bakery_day_forecast", "bakery_day_forecast_bias_adj"])
    base = metric_df[metric_df["model"] == "bakery_day_forecast"].iloc[0]
    adj = metric_df[metric_df["model"] == "bakery_day_forecast_bias_adj"].iloc[0]
    bakery_metrics.append(
        {
            "bakery_id": bakery_id,
            "bakery_name": group["bakery_name"].iloc[0],
            "city": group["city"].iloc[0],
            "base_mae": base["mae"],
            "adj_mae": adj["mae"],
            "mae_delta": adj["mae"] - base["mae"],
            "base_bias": base["bias"],
            "adj_bias": adj["bias"],
            "abs_bias_delta": abs(adj["bias"]) - abs(base["bias"]),
        }
    )

bakery_metrics = pd.DataFrame(bakery_metrics)
print("Bias-adjusted forecast improved MAE for bakeries:", int((bakery_metrics['mae_delta'] < 0).sum()))
print("Bias-adjusted forecast reduced abs(bias) for bakeries:", int((bakery_metrics['abs_bias_delta'] < 0).sum()))
display(bakery_metrics.sort_values("mae_delta").head(20))
display(bakery_metrics.sort_values("mae_delta", ascending=False).head(20))
